In [ ]:
%load_ext autoreload
%autoreload 2

import rastera
import geopandas as gpd

# Rastera

In [ ]:
bucket = "e84-earth-search-sentinel-data"
path = "sentinel-2-c1-l2a/33/T/TG/2025/7/S2B_T33TTG_20250703T100029_L2A/B03.tif"
# https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1/collections/sentinel-2-c1-l2a/items/S2B_T33TTG_20250703T100029_L2A?.language=en&.asset=asset-red
# uri1 = "https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/sentinel-2-c1-l2a/33/T/TG/2025/7/S2B_T33TTG_20250703T100029_L2A/B04.tif"
uri = "s3://e84-earth-search-sentinel-data/sentinel-2-c1-l2a/33/T/TG/2025/7/S2B_T33TTG_20250703T100029_L2A/B03.tif"
# https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1/collections/sentinel-2-c1-l2a/items/S2B_T33TUG_20250703T100029_L2A?.language=en&.asset=asset-red
# uri2 = "https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/sentinel-2-c1-l2a/33/T/UG/2025/7/S2B_T33TUG_20250703T100029_L2A/B04.tif"
uri2 = "s3://e84-earth-search-sentinel-data/sentinel-2-c1-l2a/33/T/UG/2025/7/S2B_T33TUG_20250703T100029_L2A/B03.tif"

df_aoi = gpd.read_file("../data/sentinel2_rome_e84/rome_left_small.geojson")
df_aoi = df_aoi.to_crs(df_aoi.estimate_utm_crs())
print(df_aoi.crs.to_epsg())

bbox_utm = df_aoi.geometry.total_bounds
print(bbox_utm)

bands = [1]

df_aoi_shared = gpd.read_file("../data/sentinel2_rome_e84/rome_shared.geojson")
utm_crs = df_aoi_shared.estimate_utm_crs()
df_aoi_shared = df_aoi_shared.to_crs(utm_crs)
bbox_shared = df_aoi_shared.geometry.total_bounds

In [ ]:
src = await rastera.open(uri)
src

In [ ]:
# Alternative, share store
# from rastera import S3Store

# store = rastera.S3Store(bucket, region="us-west-2", skip_signature=True)
# src = await rastera.open(uri, store=store)
# src

In [ ]:
# Full img
array = await src.read()

print(array.shape)
print(array.transform)
print(array.data.mean())

In [ ]:
# Read bbox
src = await rastera.open(uri)

array = await src.read(bbox=bbox_utm, 
                              bbox_crs=utm_crs,
                              target_crs=32633,
                              target_resolution=20
                              )
print(array.shape)
print(array.transform)
print(array.data.mean())

In [ ]:
# Read window
window = rastera.Window(col_off=0, row_off=0, width=1000, height=2000)
array = await src.read(window=window, target_resolution=20)

print(array.shape)
print(array.transform)
print(array.data.mean())

# Merged read

In [ ]:
# Before (sequential, separate stores):
#src1 = await rastera.open(uri)
#src2 = await rastera.open(uri2)

sources = await rastera.open([uri, uri2])
print(sources)

array = await rastera.merge(sources, bbox=bbox_shared, bbox_crs=utm_crs, target_crs=int(utm_crs.to_epsg()), target_resolution=10)

print(array.shape)
print(array.transform)
print(array.data.mean())

In [ ]:
# # rasterio mergeimport rasterio
# #!pip install rasterio
# from rasterio.plot import show
# from rastera.geo import Window

# from rasterio.merge import merge
# from rasterio.warp import calculate_default_transform
# from rasterio.transform import from_bounds
# import numpy as np
# import rasterio

# with rasterio.open(uri) as r1, rasterio.open(uri2) as r2:
#     # Build the output transform from bbox + resolution
#     res = 10
#     dst_transform = from_bounds(*bbox_shared, 
#                                 width=int((bbox_shared[2] - bbox_shared[0]) / res),
#                                 height=int((bbox_shared[3] - bbox_shared[1]) / res))
    
#     rio_img, rio_transform = merge(
#         [r1, r2],
#         bounds=tuple(bbox_shared),
#         res=res,
#     )
    
#     print(rio_img.shape)
#     print(rio_transform)
#     print(rio_img.mean())

# COG header cache via geoparquet index

Pre-cache COG headers in a geoparquet file for persistent cross-session caching (~6x faster opens).

`rastera.open()` also keeps an in-memory LRU cache of parsed headers within the session (default 128, configurable via `set_cache_size()`), so repeated opens skip the network fetch even without an index.

In [ ]:
import rastera

uris = [uri, uri2]

# Build once, save to disk
gdf = await rastera.build_index(uris, region="us-west-2")
gdf.to_parquet("index.parquet")
gdf

In [ ]:
# Open from index (reusable across sessions, ~5-6x faster opens)
sources = await rastera.open_from_index("index.parquet", bbox=tuple(bbox_shared), bbox_crs=int(utm_crs.to_epsg()), region="us-west-2")
array = await rastera.merge(sources, bbox=tuple(bbox_shared), bbox_crs=int(utm_crs.to_epsg()), target_crs=int(utm_crs.to_epsg()), target_resolution=10)

print(array.shape)
print(array.transform)
print(array.data.mean())